In [1]:
pip install pandas numpy matplotlib seaborn google-api-python-client google-auth-httplib2 google-auth-oauthlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 1. IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from googleapiclient.discovery import build
#from google.colab import auth
from google.auth import default

In [3]:
# Altere este mês sempre que quiser analisar um período diferente
mes_referencia = '03/2026'

In [4]:
# 3. URLs DAS SÉRIES HISTÓRICAS (CSVs Públicos)
url_nordeste = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vSW_XDz_ND11qY6XSE2lk3LwdAYeZmyr2baCa13Xgz6IfOnIPTJuY7MRCPzstOpJhs6Fd2U7EC6YN_D/pub?gid=1972864281&single=true&output=csv'
url_fortaleza = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vQHuW62I3dvO5o0vxyULvhApE_HV91C7ljUd6hsYfC7ZoUWHY8FaluqX6M08hpt6-UiIBofz2cq-DOf/pub?gid=1066520694&single=true&output=csv'

# 4. LEITURA E FILTRAGEM INICIAL
# Lendo as bases completas
df_nordeste_raw = pd.read_csv(url_nordeste)
df_fortaleza_raw = pd.read_csv(url_fortaleza)

# Filtrando imediatamente pelo mês de referência para otimizar a memória
df_ne = df_nordeste_raw[df_nordeste_raw['Mês_Ano'] == mes_referencia].copy()
df_fort = df_fortaleza_raw[df_fortaleza_raw['Mês_Ano'] == mes_referencia].copy()

# 5. EXIBIÇÃO DE CONFIRMAÇÃO
print(f"✅ Bases de {mes_referencia} carregadas com sucesso!")
print(f"-> Registros no Nordeste (RN/PB): {len(df_ne)}")
print(f"-> Registros em Fortaleza (CE): {len(df_fort)}")

if len(df_ne) == 0 or len(df_fort) == 0:
    print(f"⚠️ ATENÇÃO: Uma das bases não possui dados para o mês {mes_referencia}. Verifique os links!")

✅ Bases de 03/2026 carregadas com sucesso!
-> Registros no Nordeste (RN/PB): 282
-> Registros em Fortaleza (CE): 3636


In [5]:
# =========================================================
# 2. TRADUTOR UNIVERSAL E PADRONIZAÇÃO DE PRODUTOS
# =========================================================


# 1. Adiciona a coluna de Estado na base de Fortaleza (CE)
df_fort['Estado'] = 'CE'

# 2. O DICIONÁRIO DE EQUIVALÊNCIA (Mapeamento dos 3 Estados Atualizado)
dicionario_tradutor = {
    # --- ARROZ ---
    'Arroz branco - tipo 1 - Camil 1 kg': 'Arroz (kg)', # CE
    'Arroz branco tipo 1 - Camil 1 Kg': 'Arroz (kg)', # RN
    'Arroz branco tipo 1 - Camil': 'Arroz (kg)', # PB

    # --- FEIJÃO ---
    'Feijão carioca - Kicaldo 1 kg': 'Feijão (kg)', # CE
    'Feijão Carioca Kicaldo - 1 Kg': 'Feijão (kg)', # RN
    'Feijão Carioca - Kicaldo': 'Feijão (kg)', # PB

    # --- CARNE BOVINA (COXÃO MOLE) ---
    'Carne bovina - chã de dentro - coxão mole 1 kg': 'Carne (Coxão Mole) (kg)', # CE
    'Carne bovina - Chã de dentro (coxão mole) 1 Kg': 'Carne (Coxão Mole) (kg)', # RN
    'Carne Bovina - Chã de Dentro (Coxão Mole)': 'Carne (Coxão Mole) (kg)', # PB

    # --- CARNE BOVINA (OUTROS CORTES) ---
    'Carne bovina - chã de fora - coxão duro 1 kg': 'Carne (Coxão Duro) (kg)', # CE
    'Carne Bovina - Chã de Fora (Coxão Duro)': 'Carne (Coxão Duro) (kg)', # PB
    'Carne Bovina - Chambaril (Ossobuco)': 'Carne (Chambaril) (kg)', # PB
    'Carne bovina 2ª - lombo com osso 1 kg': 'Carne (Lombo com Osso) (kg)', # CE

    # --- LEITE ---
    'Leite longa vida - integral - Betânia 1 l': 'Leite (L)', # CE
    'Leite integral - Betânia 1 L': 'Leite (L)', # RN
    'Leite integral - Betânia': 'Leite (L)', # PB

    # --- PÃO FRANCÊS ---
    'Pão francês - 1 kg': 'Pão Francês (kg)', # CE
    'Pão Francês - 1kg': 'Pão Francês (kg)', # RN (Ajustado o F maiúsculo)
    'Pão Francês': 'Pão Francês (kg)', # PB

    # --- TOMATE ---
    'Tomate - 1 kg': 'Tomate (kg)', # CE
    'Tomate comum - 1 kg': 'Tomate (kg)', # RN (Ajustado de Tomate de Salada)
    'Tomate': 'Tomate (kg)', # PB

    # --- CAFÉ ---
    'Café em pó - clássico - Santa Clara 250 g': 'Café (250g)', # CE
    'Café em pó - Santa Clara 250 g': 'Café (250g)', # RN e PB

    # --- AÇÚCAR ---
    'Açúcar cristal - menor valor 1 kg': 'Açúcar (kg)', # CE
    'Açúcar Estrela 1 Kg': 'Açúcar (kg)', # RN (Novo!)
    'Açúcar - Alegre': 'Açúcar (kg)', # PB

    # --- FARINHA DE MANDIOCA ---
    'Farinha de mandioca fina - branca - menor valor 1 kg': 'Farinha (kg)', # CE
    'Farinha de Mandioca Kicaldo 1 Kg': 'Farinha (kg)', # RN
    'Farinha de mandioca fina branca - Kicaldo': 'Farinha (kg)', # PB

    # --- ÓLEO DE SOJA ---
    'Óleo de soja - Soya 900 ml': 'Óleo de Soja (900ml)', # CE
    'Óleo Soja Soya 900 ml ': 'Óleo de Soja (900ml)', # RN (Novo! Com espaço no final conforme a planilha)
    'Óleo Soja Soya 900 ml': 'Óleo de Soja (900ml)', # RN (Garantia caso o espaço suma)
    'Óleo de soja - Soya': 'Óleo de Soja (900ml)', # PB

    # --- BANANA ---
    'Banana - prata 1 kg': 'Banana (kg)', # CE
    'Banana Pacovã - 1 kg': 'Banana (kg)', # RN (Novo!)
    'Banana - Prata': 'Banana (kg)', # PB

    # --- MANTEIGA ---
    'Manteiga - menor valor 500 g': 'Manteiga', # CE (500g)
    'Manteiga Itacolomy 200g': 'Manteiga', # RN (200g)
    'Manteiga Itacolomy': 'Manteiga' # PB
}

# 3. Cria a coluna de "Produto Comparável" mapeando os dicionários (limpando espaços extras por segurança)
df_ne['Produto_Cesta'] = df_ne['Produto'].map(dicionario_tradutor)
df_fort['Produto_Cesta'] = df_fort['Produto'].map(dicionario_tradutor)

# 4. Mantém apenas os produtos que conseguimos "traduzir" (parear entre os estados)
df_ne_cesta = df_ne.dropna(subset=['Produto_Cesta']).copy()
df_fort_cesta = df_fort.dropna(subset=['Produto_Cesta']).copy()

print("✅ Tradutor aplicado com sucesso com as novas nomenclaturas!")
print(f"-> Itens pareados no Nordeste (RN/PB): {df_ne_cesta['Produto_Cesta'].nunique()}")
print(f"-> Itens pareados em Fortaleza (CE): {df_fort_cesta['Produto_Cesta'].nunique()}")

✅ Tradutor aplicado com sucesso com as novas nomenclaturas!
-> Itens pareados no Nordeste (RN/PB): 14
-> Itens pareados em Fortaleza (CE): 14


In [6]:
# =========================================================
# 3. CÁLCULO DA CESTA E AUDITORIA DE FALTAS NA RAIZ
# =========================================================

# 1. Junta as bases (Nordeste e Fortaleza)
df_regional = pd.concat([df_ne_cesta, df_fort_cesta], ignore_index=True)

# Garante que os preços são números
df_regional['Preço'] = pd.to_numeric(df_regional['Preço'].astype(str).str.replace(',', '.'), errors='coerce')

# --- LÓGICA DE ASTERISCOS DA AYURI (LIDA DIRETAMENTE DA SÉRIE HISTÓRICA) ---
# Quantos produtos cada estado pesquisa oficialmente em suas planilhas?
produtos_oficiais = {'CE': 14, 'PB': 14, 'RN': 12}

# Filtra apenas os preços que existem (não nulos) e conta quantos produtos sobreviveram
df_validos = df_regional.dropna(subset=['Preço'])
produtos_lidos = df_validos.groupby('Estado')['Produto_Cesta'].nunique()

# Calcula a diferença (As faltas reais na série histórica)
asteriscos_estado = {}
for estado, esperado in produtos_oficiais.items():
    lidos = produtos_lidos.get(estado, 0)
    faltas = esperado - lidos
    asteriscos_estado[estado] = max(0, faltas) # Guarda o número de asteriscos

print("📊 AUDITORIA DE PRODUTOS LIDOS NA SÉRIE HISTÓRICA:")
print(f"-> Ceará: Esperados {produtos_oficiais['CE']} | Lidos: {produtos_lidos.get('CE', 0)} | Faltas (Asteriscos): {asteriscos_estado['CE']}")
print(f"-> Paraíba: Esperados {produtos_oficiais['PB']} | Lidos: {produtos_lidos.get('PB', 0)} | Faltas (Asteriscos): {asteriscos_estado['PB']}")
print(f"-> Rio G. do Norte: Esperados {produtos_oficiais['RN']} | Lidos: {produtos_lidos.get('RN', 0)} | Faltas (Asteriscos): {asteriscos_estado['RN']}")
print("-" * 50)

# 2. O MOTOR DE CÁLCULO DA CESTA BÁSICA
def calcular_valor_cesta(row):
    produto = row['Produto_Cesta']
    preco = row['Preço']
    estado = row['Estado']

    # --- CARNES (SUAS REGRAS EXATAS) ---
    if 'Carne' in produto:
        if estado == 'PB': return preco * 1.0   # PB: Apenas soma os cortes
        elif estado == 'CE': return preco * 1.5 # CE: 1,5kg de cada um dos 3 cortes
        elif estado == 'RN': return preco * 4.5 # RN: O único corte multiplica por 4,5

    # --- MANTEIGA ---
    elif produto == 'Manteiga':
        if estado in ['CE', 'PB']: return preco * 1.5
        elif estado == 'RN': return preco * 3.7

    # --- CAFÉ ---
    elif produto == 'Café (250g)': return (preco / 250.0) * 300.0

    # --- OUTROS PRODUTOS ---
    elif produto == 'Arroz (kg)': return preco * 3.6
    elif produto == 'Banana (kg)': return preco * 1.0
    elif produto == 'Feijão (kg)': return preco * 4.5
    elif produto == 'Leite (L)': return preco * 6.0
    elif produto == 'Óleo de Soja (900ml)': return preco * 1.0
    elif produto == 'Pão Francês (kg)': return preco * 6.0
    elif produto == 'Tomate (kg)': return preco * 12.0
    elif produto == 'Açúcar (kg)': return preco * 3.0
    elif produto == 'Farinha (kg)': return preco * 3.0

    return preco

# Aplica o cálculo usando apenas a base limpa
df_validos['Valor_Mensal_Cesta'] = df_validos.apply(calcular_valor_cesta, axis=1)

# Calcula as médias finais
df_ranking = df_validos.groupby(['Produto_Cesta', 'Estado'])['Valor_Mensal_Cesta'].mean().unstack()
df_ranking = df_ranking.rename(columns={'CE': 'Fortaleza', 'RN': 'Natal', 'PB': 'João Pessoa'})

# Traduz as siglas para usar no gráfico
asteriscos_capital = {
    'Fortaleza': asteriscos_estado['CE'],
    'Natal': asteriscos_estado['RN'],
    'João Pessoa': asteriscos_estado['PB']
}
print("✅ Valores da Cesta Básica calculados com sucesso!")

📊 AUDITORIA DE PRODUTOS LIDOS NA SÉRIE HISTÓRICA:
-> Ceará: Esperados 14 | Lidos: 14 | Faltas (Asteriscos): 0
-> Paraíba: Esperados 14 | Lidos: 13 | Faltas (Asteriscos): 1
-> Rio G. do Norte: Esperados 12 | Lidos: 12 | Faltas (Asteriscos): 0
--------------------------------------------------
✅ Valores da Cesta Básica calculados com sucesso!


In [ ]:
# =========================================================
# 4. O CARRINHO DE COMPRAS OTIMIZADO (ECONOMIA MÁXIMA)
# =========================================================

# 1. Encontra o menor preço de cada item por cidade (PDV mais barato)
df_carrinho = df_validos.groupby(['Estado', 'Produto_Cesta'])['Preço'].min().reset_index()

# 2. Aplica a mesma regra de cálculo da cesta sobre os preços mínimos
df_carrinho['Valor_Cesta_Minimo'] = df_carrinho.apply(calcular_valor_cesta, axis=1)

# 3. Consolida o Valor Total do "Carrinho Inteligente" por Capital
carrinho_por_capital = df_carrinho.groupby('Estado')['Valor_Cesta_Minimo'].sum()

# 4. Compara com a Média (que você já calculou no df_ranking)
media_por_capital = df_ranking.sum() # Soma das médias de todos os produtos por coluna

# 5. Cria o DataFrame de Economia Real
df_economia = pd.DataFrame({
    'Cesta Média (R$)': [media_por_capital['Fortaleza'], media_por_capital['Natal'], media_por_capital['João Pessoa']],
    'Carrinho Otimizado (R$)': [carrinho_por_capital.get('CE', 0), carrinho_por_capital.get('RN', 0), carrinho_por_capital.get('PB', 0)]
}, index=['Fortaleza', 'Natal', 'João Pessoa'])

df_economia['Economia Estimada (R$)'] = df_economia['Cesta Média (R$)'] - df_economia['Carrinho Otimizado (R$)']
df_economia['% de Economia'] = (df_economia['Economia Estimada (R$)'] / df_economia['Cesta Média (R$)']) * 100

print("🛒 CARRINHO DE COMPRAS - POTENCIAL DE ECONOMIA:")
print(df_economia.round(2))

🛒 CARRINHO DE COMPRAS - POTENCIAL DE ECONOMIA:
             Cesta Média (R$)  Carrinho Otimizado (R$)  \
Fortaleza              603.46                   410.54   
Natal                  605.16                   449.94   
João Pessoa            538.67                   498.49   

             Economia Estimada (R$)  % de Economia  
Fortaleza                    192.92          31.97  
Natal                        155.22          25.65  
João Pessoa                   40.18           7.46  


In [ ]:
import json

# 1. Dicionário de Emojis para deixar a vitrine mais visual
emojis = {
    'Arroz (kg)': '🍚', 'Feijão (kg)': '🫘', 'Carne (Coxão Mole) (kg)': '🥩',
    'Carne (Coxão Duro) (kg)': '🥩', 'Carne (Chambaril) (kg)': '🍖', 'Carne (Lombo com Osso) (kg)': '🍖',
    'Leite (L)': '🥛', 'Pão Francês (kg)': '🥖', 'Tomate (kg)': '🍅',
    'Café (250g)': '☕', 'Açúcar (kg)': '🧂', 'Farinha (kg)': '🥣',
    'Óleo de Soja (900ml)': '🛢️', 'Banana (kg)': '🍌', 'Manteiga': '🧈'
}

# 2. Estrutura vazia para os três estados
catalogo_regional = {
    "fortaleza": [],
    "natal": [],
    "joaopessoa": []
}

# Traduz a sigla do estado para a chave do nosso JSON
mapa_estados = {'CE': 'fortaleza', 'RN': 'natal', 'PB': 'joaopessoa'}

# 3. Varre o DataFrame de menores preços (df_carrinho) e preenche o catálogo
for index, row in df_carrinho.iterrows():
    estado_sigla = row['Estado']
    cidade_chave = mapa_estados.get(estado_sigla)
    
    if cidade_chave:
        produto_nome = row['Produto_Cesta']
        # Cria um ID único sem espaços ou parênteses
        produto_id = produto_nome.replace(' ', '_').replace('(', '').replace(')', '').lower()
        
        catalogo_regional[cidade_chave].append({
            "id": produto_id,
            "nome": produto_nome,
            "preco": round(row['Preço'], 2),
            "imagem": emojis.get(produto_nome, "🛒") # Pega o emoji ou usa o carrinho por padrão
        })

# 4. Salva o super arquivo JSON
with open('catalogo.json', 'w', encoding='utf-8') as f:
    json.dump(catalogo_regional, f, ensure_ascii=False, indent=4)

print("✅ Catálogo Regional exportado com sucesso para as 3 capitais!")

✅ Catálogo Regional exportado com sucesso para as 3 capitais!


In [12]:
# 1. Dicionário completo de Emojis para todos os itens da cesta
emojis = {
    'Arroz (kg)': '🍚', 
    'Feijão (kg)': '🫘', 
    'Carne (Coxão Mole) (kg)': '🥩',
    'Carne (Coxão Duro) (kg)': '🥩', 
    'Carne (Chambaril) (kg)': '🍖', 
    'Carne (Lombo com Osso) (kg)': '🍖',
    'Leite (L)': '🥛', 
    'Pão Francês (kg)': '🥖', 
    'Tomate (kg)': '🍅',
    'Café (250g)': '☕', 
    'Açúcar (kg)': '🧂', 
    'Farinha (kg)': '🥣',
    'Óleo de Soja (900ml)': '🛢️', 
    'Banana (kg)': '🍌', 
    'Manteiga': '🧈'
}

# 2. Estrutura vazia para os três estados
catalogo_regional = {
    "fortaleza": [],
    "natal": [],
    "joaopessoa": []
}

# Traduz a sigla do estado para a chave do JSON
mapa_estados = {'CE': 'fortaleza', 'RN': 'natal', 'PB': 'joaopessoa'}

# 3. Preenche o catálogo com os menores preços achados (df_carrinho)
for index, row in df_carrinho.iterrows():
    estado_sigla = row['Estado']
    cidade_chave = mapa_estados.get(estado_sigla)
    
    if cidade_chave:
        produto_nome = row['Produto_Cesta']
        # Cria um ID único
        produto_id = produto_nome.replace(' ', '_').replace('(', '').replace(')', '').lower()
        
        catalogo_regional[cidade_chave].append({
            "id": produto_id,
            "nome": produto_nome,
            "preco": round(row['Preço'], 2),
            "imagem": emojis.get(produto_nome, "🛒") # Garante o emoji ou um carrinho padrão
        })

# 4. Salva o arquivo JSON na mesma pasta
with open('catalogo.json', 'w', encoding='utf-8') as f:
    json.dump(catalogo_regional, f, ensure_ascii=False, indent=4)

print("✅ Arquivo 'catalogo.json' gerado com sucesso com todos os itens!")

✅ Arquivo 'catalogo.json' gerado com sucesso com todos os itens!


In [16]:
import json
import pandas as pd

emojis = {
    'Arroz (kg)': '🍚', 'Feijão (kg)': '🫘', 'Carne (Coxão Mole) (kg)': '🥩',
    'Carne (Coxão Duro) (kg)': '🥩', 'Carne (Chambaril) (kg)': '🍖', 'Carne (Lombo com Osso) (kg)': '🍖',
    'Leite (L)': '🥛', 'Pão Francês (kg)': '🥖', 'Tomate (kg)': '🍅',
    'Café (250g)': '☕', 'Açúcar (kg)': '🧂', 'Farinha (kg)': '🥣',
    'Óleo de Soja (900ml)': '🛢️', 'Banana (kg)': '🍌', 'Manteiga': '🧈'
}

# Estrutura temporária baseada em dicionários para facilitar o agrupamento
catalogo_temp = {"fortaleza": {}, "natal": {}, "joaopessoa": {}}
mapa_estados = {'CE': 'fortaleza', 'RN': 'natal', 'PB': 'joaopessoa'}
coluna_mercado = 'Supermercado' 

# 1. Agrupa todas as ofertas por Produto
for index, row in df_validos.iterrows():
    cidade_chave = mapa_estados.get(row['Estado'])
    nome_mercado = str(row[coluna_mercado]).strip()
    
    if cidade_chave and nome_mercado.lower() != 'nan':
        produto_nome = row['Produto_Cesta']
        produto_id = produto_nome.replace(' ', '_').replace('(', '').replace(')', '').lower()
        
        # Se o produto ainda não existe na cidade, cria a estrutura dele
        if produto_id not in catalogo_temp[cidade_chave]:
            catalogo_temp[cidade_chave][produto_id] = {
                "id": produto_id,
                "nome": produto_nome,
                "imagem": emojis.get(produto_nome, "🛒"),
                "ofertas": [] # Lista que vai guardar todos os preços
            }
        
        # Adiciona a oferta do mercado atual na lista do produto
        catalogo_temp[cidade_chave][produto_id]["ofertas"].append({
            "mercado": nome_mercado,
            "preco": round(row['Preço'], 2)
        })

# 2. Formata para o JSON final e ordena as ofertas da mais barata para a mais cara
catalogo_final = {"fortaleza": [], "natal": [], "joaopessoa": []}

for cidade, produtos in catalogo_temp.items():
    for prod_id, dados_produto in produtos.items():
        # Remove ofertas duplicadas (caso o mesmo mercado apareça duas vezes na base pro mesmo item)
        ofertas_unicas = {oferta['mercado']: oferta for oferta in dados_produto['ofertas']}.values()
        
        # Ordena a lista de ofertas do menor preço para o maior
        ofertas_ordenadas = sorted(ofertas_unicas, key=lambda x: x['preco'])
        dados_produto['ofertas'] = ofertas_ordenadas
        
        catalogo_final[cidade].append(dados_produto)

# 3. Salva o arquivo
with open('catalogo.json', 'w', encoding='utf-8') as f:
    json.dump(catalogo_final, f, ensure_ascii=False, indent=4)

print("✅ Arquivo 'catalogo.json' atualizado com o ranking de ofertas por produto!")

✅ Arquivo 'catalogo.json' atualizado com o ranking de ofertas por produto!
